<a href="https://colab.research.google.com/github/detektor777/colab_list_image/blob/main/odtsr_qwen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ##**Install** { display-mode: "form" }
show_log = False #@param {type:"boolean"}

def log(*args, **kwargs):
    if show_log:
        print(*args, **kwargs)

# Everything here runs once per session: best-effort swap (optional — model loading
# now uses zero-copy mmap and fits in free Colab's ~13 GB RAM), environment setup,
# patches, and downloading all weights. Nothing depends on the images/settings below.
import os
# reduce CUDA memory fragmentation (helps against borderline OOM in tiled inference)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import subprocess, shutil, torch, psutil

# --- GPU / RAM check ---
n_gpu = torch.cuda.device_count()
log(f'PyTorch: {torch.__version__}, GPU count: {n_gpu}')
assert n_gpu >= 1, 'No GPU! Runtime -> Change runtime type -> T4 GPU'
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
log(f'  cuda:0 -> {torch.cuda.get_device_properties(0).name}, {GPU_VRAM_GB:.1f} GB VRAM')
RAM_GB = psutil.virtual_memory().total / 1e9
log(f'  RAM: {RAM_GB:.1f} GB')
assert hasattr(torch, 'float8_e4m3fn'), 'torch >= 2.1 is required (already available on Colab)'

# --- Swap file: BEST-EFFORT and now OPTIONAL ---
# Model loading uses zero-copy mmap (see cell 4), so peak RAM is ~3-5 GB and swap
# is no longer required on free Colab. We still try to enable it as extra headroom,
# but Colab containers usually run without CAP_SYS_ADMIN, so `swapon` fails with
# 'Operation not permitted' — in that case we just carry on.
total_d, used_d, free_d = shutil.disk_usage('/content')
free_gb = free_d / 1e9
# weights ~55 GB + repo/pip ~6 GB must also fit on this disk
SWAP_GB = max(8, min(20, int(free_gb - 64)))
SWAP_OK = 'swapfile' in open('/proc/swaps').read()
if not SWAP_OK and RAM_GB < 26:
    try:
        log(f'\nTrying to enable {SWAP_GB} GB swap (disk free: {free_gb:.0f} GB)...')
        subprocess.run(['fallocate', '-l', f'{SWAP_GB}G', '/content/swapfile'], check=True)
        subprocess.run(['chmod', '600', '/content/swapfile'], check=True)
        subprocess.run(['mkswap', '/content/swapfile'], check=True)
        subprocess.run(['swapon', '/content/swapfile'], check=True)
        SWAP_OK = 'swapfile' in open('/proc/swaps').read()
        log('Swap active:', open('/proc/swaps').read())
    except Exception as e:
        # Expected on free Colab: drop the unused file so it doesn't waste disk.
        subprocess.run(['rm', '-f', '/content/swapfile'])
        log(f'[!] Swap not permitted ({type(e).__name__}) — normal on free Colab. '
              'Continuing without swap.')
elif SWAP_OK:
    log('Swap already active.')
else:
    log('RAM >= 26 GB (Pro high-RAM runtime) — swap not needed.')

# RAM note for the model-load step (cell 4).
if not SWAP_OK and RAM_GB < 26:
    log(
        '\n' + '=' * 70 + '\n'
        'NOTE: cell 4 loads the 20B DiT via zero-copy mmap (bf16, no dtype\n'
        'conversion on CPU), so peak RAM stays at ~3-5 GB and free Colab\n'
        '(~13 GB RAM) is enough. Swap is NOT required. If the session still\n'
        'crashes for RAM in cell 4, make sure the Install cell re-applied the\n'
        'generator.py patches (use_fp8 must be env-controlled and OFF).\n'
        + '=' * 70)

# --- Clone the repository ---
WORK = '/content'
ODTSR_DIR = f'{WORK}/ODTSR'
if not os.path.exists(ODTSR_DIR):
    # capture_output: git's own clone progress used to print straight to the cell
    # regardless of show_log, since it bypasses the log() function entirely.
    r = subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/RedMediaTech/ODTSR.git', ODTSR_DIR],
                       capture_output=True, text=True)
    if show_log:
        print(r.stderr)  # git clone writes its progress to stderr
    if r.returncode != 0:
        print(r.stderr)  # always show on failure, even with show_log off
        raise RuntimeError('git clone failed - see the log above.')

# --- Dependencies ---
# Colab ships newer transformers by default — diffsynth needs the pinned 4.56.0.
# bitsandbytes provides the INT4 (NF4) / INT8 quantization used in the Config cell.
# subprocess instead of the %pip magic: the magic always streams to the cell
# output regardless of show_log. capture_output lets us gate it the same way.
import sys
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'einops', 'safetensors', 'sentencepiece', 'protobuf', 'ftfy', 'transformers==4.56.0',
     'accelerate', 'lightning', 'peft', 'imageio', 'pandas', 'huggingface_hub',
     'hf_transfer', 'ipywidgets', 'bitsandbytes'],
    capture_output=True, text=True)
if show_log:
    print(r.stdout[-3000:])
if r.returncode != 0:
    print(r.stdout[-3000:]); print(r.stderr[-3000:])
    raise RuntimeError('pip install failed - see the log above.')

# --- Patch for incompatible transformers imports in diffsynth ---
step_path = f'{ODTSR_DIR}/diffsynth/models/stepvideo_text_encoder.py'
if os.path.exists(step_path):
    src = open(step_path).read()
    old_imp = 'from transformers.modeling_utils import PretrainedConfig, PreTrainedModel'
    if old_imp in src:
        src = src.replace(
            old_imp,
            'from transformers.modeling_utils import PreTrainedModel\n'
            'try:\n'
            '    from transformers import PretrainedConfig\n'
            'except ImportError:\n'
            '    from transformers import PreTrainedConfig as PretrainedConfig')
        open(step_path, 'w').write(src)
        log('stepvideo_text_encoder.py: import patch applied')

# --- modelscope stub: shuts down ALL modelscope imports in diffsynth at once ---
import sysconfig
site_dir = sysconfig.get_paths()['purelib']
stub_dir = os.path.join(site_dir, 'modelscope')
os.makedirs(os.path.join(stub_dir, 'hub'), exist_ok=True)
with open(os.path.join(stub_dir, '__init__.py'), 'w') as f:
    f.write(
        'def snapshot_download(*a, **k):\n'
        '    raise RuntimeError("modelscope stub: use HuggingFace instead")\n'
        'def dataset_snapshot_download(*a, **k):\n'
        '    raise RuntimeError("modelscope stub: use HuggingFace instead")\n'
    )
with open(os.path.join(stub_dir, 'hub', '__init__.py'), 'w') as f:
    f.write('')
with open(os.path.join(stub_dir, 'hub', 'api.py'), 'w') as f:
    f.write(
        'class HubApi:\n'
        '    def __getattr__(self, name):\n'
        '        raise RuntimeError("modelscope stub: use HuggingFace instead")\n'
    )
log('modelscope stub installed at', stub_dir)

# --- Patch generator.py: FP8 storage becomes OPT-IN via an env variable ---
# CRITICAL for free Colab: any dtype conversion on CPU (bf16 -> fp8) MATERIALIZES
# ~20 GB in RAM and kills the session. Keeping bf16 preserves safetensors' zero-copy
# mmap: the DiT 'lives' on disk and RSS stays at ~2-3 GB during the whole load.
gen_path = f'{ODTSR_DIR}/examples/qwen_image/generator.py'
src = open(gen_path).read()
changed = False
if "use_offload = os.environ.get" not in src:
    src = src.replace('use_offload = False', "use_offload = os.environ.get('ODTSR_FP8', '0') == '1'")
    changed = True
if "use_fp8 = os.environ.get" not in src:
    # hardcoded use_fp8=True converts EVERY wrapped Linear weight to fp8 on CPU -> +20 GB RAM
    src = src.replace('use_fp8 = True', "use_fp8 = (os.environ.get('ODTSR_FP8', '0') == '1')")
    changed = True
if '"seed": 42' in src:
    src = src.replace('"seed": 42', '"seed": int(os.environ.get("ODTSR_SEED", "42"))')
    changed = True
if changed:
    open(gen_path, 'w').write(src)
    log('generator.py: FP8 (opt-in), use_fp8 and seed patches applied')
else:
    log('generator.py: patches already applied')
assert "use_fp8 = (os.environ.get" in open(gen_path).read(), 'use_fp8 patch failed to apply!'

# --- Patch model_training/train.py: disable the training-dataset import ---
train_path = f'{ODTSR_DIR}/examples/qwen_image/model_training/train.py'
src = open(train_path).read()
ds_imp = 'from diffsynth.extensions.realesrgan.dataset import PairedSROnlineTxtDataset'
if ds_imp in src:
    src = src.replace(ds_imp, 'PairedSROnlineTxtDataset = None  # patched: training dataset not needed for inference')
    open(train_path, 'w').write(src)
    log('train.py: training-dataset import disabled')
else:
    log('train.py: patch already applied')

# --- Patch model_manager.py: convert dtype shard-by-shard while loading ---
mm_path = f'{ODTSR_DIR}/diffsynth/models/model_manager.py'
src = open(mm_path).read()
old_load = 'state_dict.update(load_state_dict(path))'
if old_load in src:
    src = src.replace(old_load, 'state_dict.update(load_state_dict(path, torch_dtype=torch_dtype))')
    open(mm_path, 'w').write(src)
    log('model_manager.py: streaming dtype-conversion patch applied')
else:
    log('model_manager.py: patch already applied')

import sys
for p in (ODTSR_DIR, f'{ODTSR_DIR}/examples/qwen_image'):
    if p not in sys.path:
        sys.path.insert(0, p)

log('\nEnvironment ready. Downloading weights (~55 GB, 15-30 min)...\n')

# --- Download weights (Qwen-Image + ODTSR checkpoint) ---
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
if not show_log:
    # snapshot_download/hf_hub_download print tqdm progress bars straight to the
    # cell regardless of show_log - silence them explicitly when the log is off.
    os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
from huggingface_hub import snapshot_download, hf_hub_download, list_repo_files

MODELS_DIR = f'{WORK}/models'
QWEN_DIR   = f'{MODELS_DIR}/Qwen-Image'
os.makedirs(MODELS_DIR, exist_ok=True)

# huggingface_hub warns (via the `warnings` module, not print - bypasses log()/
# show_log same as git/pip did) that no HF_TOKEN secret is set. Harmless for these
# public repos; only shown when show_log is on.
import warnings
with warnings.catch_warnings():
    if not show_log:
        warnings.simplefilter('ignore')

    # --- Qwen-Image: only the needed subfolders ---
    snapshot_download(
        repo_id='Qwen/Qwen-Image',
        local_dir=QWEN_DIR,
        max_workers=8,
        allow_patterns=[
            'transformer/*.safetensors',
            'text_encoder/*.safetensors',
            'vae/diffusion_pytorch_model.safetensors',
            'tokenizer/*',
            '*.json',
        ],
    )
    log('Qwen-Image downloaded.')

    # --- ODTSR checkpoint (DualLoRA generator) ---
    # In the double8fun/ODTSR repo:
    #   weight.pth              (2.46 GB) — the GENERATOR, this is the one we need
    #   net_dis_iter_5001.pth   (2.84 GB) — discriminator, NOT needed for inference
    ODTSR_REPO = 'double8fun/ODTSR'
    GEN_CKPT_NAME = 'weight.pth'

    repo_files = list_repo_files(ODTSR_REPO)
    assert GEN_CKPT_NAME in repo_files, (
        f'{GEN_CKPT_NAME} not found in {ODTSR_REPO}. Available .pth files: '
        f'{[f for f in repo_files if f.endswith(".pth")]}'
    )
    CKPT_PATH = hf_hub_download(ODTSR_REPO, GEN_CKPT_NAME, local_dir=MODELS_DIR)
    log(f'ODTSR generator checkpoint: {CKPT_PATH} ({os.path.getsize(CKPT_PATH)/1e9:.2f} GB)')
    assert 'net_dis' not in os.path.basename(CKPT_PATH), 'Downloaded the discriminator instead of the generator!'

total, used, free = shutil.disk_usage(WORK)
log(f'Disk /content: used {used/1e9:.0f} GB, free {free/1e9:.0f} GB')

print('\nInstall complete.')

In [ ]:
#@title ##**Upload Images** { display-mode: "form" }
# Upload files with the Colab dialog below.
# Running this cell wipes any previous input/output files first.

import os, shutil

INPUT_DIR = '/content/inputs'
OUTPUT_DIR = '/content/outputs'

def _clear_dir(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

_clear_dir(INPUT_DIR)
_clear_dir(OUTPUT_DIR)

EXTS = ('.png', '.jpg', '.jpeg', '.webp')

from google.colab import files
try:
    uploaded = files.upload()  # blocking dialog; Cancel -> empty dict
except Exception as e:
    uploaded = {}
    print(f'(upload dialog skipped: {e})')

for name, content in uploaded.items():
    if not name.lower().endswith(EXTS):
        print(f'Skipping {name}: unsupported extension')
        continue
    with open(os.path.join(INPUT_DIR, os.path.basename(name)), 'wb') as out:
        out.write(content)

listing = sorted(f for f in os.listdir(INPUT_DIR) if f.lower().endswith(EXTS))
print(f'\nQueued for upscale ({len(listing)} uploaded): ' + ', '.join(listing[:20]))
assert listing, 'No images! Upload files.'

In [ ]:
#@title ##**Config** { display-mode: "form" }
prompt = 'raw photo, extremely detailed, sharp focus, realistic texture, high contrast, 8k UHD, photorealistic' #@param {type:"string"}
negative_prompt = '' #@param {type:"string"}

quantization = 'INT4 (NF4) — fits on one T4, recommended' #@param ['INT4 (NF4) — fits on one T4, recommended', 'INT8 — better quality, needs 24GB+ GPU (L4/A100)', 'BF16 + CPU offload — full quality, 5-10x slower']
SENSITIVE_INT8 = 'img_mod+txt_mod' #@param ['img_mod+txt_mod', 'img_mod', 'none']

scale = 2 #@param {type:"slider", min:1.0, max:4.0, step:0.5}
fidelity = 1.0 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
cfg = 1.0 #@param {type:"slider", min:1.0, max:4.0, step:0.5}
color_fix = 'wavelet' #@param ['wavelet', 'adain', 'no']
seed = 42 #@param {type:"integer"}

encoder = 'GPU BF16 streaming (bit-exact, recommended)' #@param ['GPU BF16 streaming (bit-exact, recommended)', 'GPU 4-bit (fast, degrades quality)', 'CPU (bit-exact, needs High-RAM)']

vae_encoder_tiled_size = '512 (default)' #@param ['1024', '768', '512 (default)', '448', '384', '320', '256', '192', '128']
vae_decoder_tiled_size = '64 (default)' #@param ['96', '80', '64 (default)', '48', '40', '32', '24']
latent_tiled_size = '64 (default)' #@param ['96', '80', '64 (default)', '48', '40', '32', '24']
latent_tiled_overlap = '16 (default)' #@param ['8', '16 (default)', '24', '32']

show_log = False #@param {type:"boolean"}

def _tile_num(v):
    return int(str(v).split()[0])

TILE_CFG = {
    'vae_enc': _tile_num(vae_encoder_tiled_size),   # pixels
    'vae_dec': _tile_num(vae_decoder_tiled_size),   # latent units
    'latent':  _tile_num(latent_tiled_size),        # latent units
    'overlap': _tile_num(latent_tiled_overlap),     # latent units
}
assert TILE_CFG['overlap'] < TILE_CFG['latent'], 'latent_tiled_overlap must be smaller than latent_tiled_size'
assert TILE_CFG['overlap'] < TILE_CFG['vae_dec'], 'latent_tiled_overlap must be smaller than vae_decoder_tiled_size'

AUTO_TILES = None  # reset the auto-tune result whenever Config changes

def current_tiles():
    """Tiles the pipeline actually uses: auto-tuned if available, else Config."""
    return dict(AUTO_TILES) if AUTO_TILES else dict(TILE_CFG)


import torch

QUANT_MODE = {
    'INT4 (NF4) — fits on one T4, recommended': 'int4',
    'INT8 — better quality, needs 24GB+ GPU (L4/A100)': 'int8',
    'BF16 + CPU offload — full quality, 5-10x slower': 'fp8_offload',
}[quantization]
ENCODE_MODE = ('gpu_stream' if encoder.startswith('GPU BF16')
               else 'gpu4bit' if encoder.startswith('GPU 4-bit') else 'cpu')

_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
_est = {
    'int4':        ('~11 GB',        'full speed',   'slightly softer fine texture'),
    'int8':        ('~21 GB',        'full speed',   'near-full quality'),
    'fp8_offload': ('~13 GB + mmap', '5-10x slower', 'full (bf16, no quantization)'),
}[QUANT_MODE]
print(f'Quantization: {QUANT_MODE}  |  VRAM: {_est[0]}  |  speed: {_est[1]}  |  quality: {_est[2]}')
print(f'GPU: {torch.cuda.get_device_properties(0).name}, {_vram:.1f} GB')

if QUANT_MODE == 'int8' and _vram < 21.5:
    raise RuntimeError(
        f'INT8 needs ~21 GB VRAM but this GPU has {_vram:.1f} GB. '
        'Choose INT4 (fits on a T4) or BF16 + CPU offload instead.')
print(f'scale x{scale}, color_fix={color_fix}, seed={seed}, fidelity={fidelity}, cfg={cfg}')
print(f'prompt: {prompt!r}')
print(f'negative: {negative_prompt!r}')
print(f"tiles: vae_enc={TILE_CFG['vae_enc']}px, vae_dec={TILE_CFG['vae_dec']}, latent={TILE_CFG['latent']}, overlap={TILE_CFG['overlap']}")

# ============================================================
# Load the model. Encodes the prompt (only if it changed) and loads/(re)quantizes the
# DiT onto the GPU only if quantization/SENSITIVE_INT8 changed since the last time this
# cell ran — otherwise this whole section is a fast no-op. Run Auto-tune tiles / Run
# after this completes.
# ============================================================
import os, gc, json, types, glob, time, subprocess, contextlib, io
from PIL import Image

# Defensive VRAM cleanup: a previous OOM/exception in this session can leave GPU
# tensors alive via Jupyter's exception traceback (IPython caches sys.last_traceback,
# which pins every local variable of every stack frame - including any tensor that was
# mid-construction when the error happened). Plain gc.collect()/empty_cache() often
# can't reclaim that memory until the traceback itself is dropped, so clear it first.
import sys
sys.last_traceback = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    _free, _total = torch.cuda.mem_get_info()
    print(f'GPU memory: {_free/1e9:.2f} GB free / {_total/1e9:.2f} GB total')

# ============================================================
# 1) Encode prompt (skipped if already cached)
# ============================================================
PROMPT_CACHE_PATH = f'/content/prompt_cache_{ENCODE_MODE}.pt'
prompts_to_encode = sorted({prompt.strip(), negative_prompt.strip(), ''})

cache = torch.load(PROMPT_CACHE_PATH, map_location='cpu') if os.path.exists(PROMPT_CACHE_PATH) else {}
missing = [p for p in prompts_to_encode if p not in cache]

if not missing:
    print(f'Prompt cache [{ENCODE_MODE}]: up to date ->', list(cache.keys()))
else:
    enc_mode = ENCODE_MODE
    if enc_mode == 'gpu4bit' and 'model' in globals():
        print('Model already on GPU - falling back to BF16 streaming for encoding.')
        enc_mode = 'gpu_stream'
    print(f'Encoding prompts ({enc_mode}, 7B encoder):', missing)
    print('(BF16 streaming: ~3-6 min, full quality. 4-bit: ~2-4 min, lossy. CPU: ~15-30 min.)')
    encoder_script = r'''
import sys, os, json, gc, torch
sys.path.insert(0, os.environ['ODTSR_DIR'])
from diffsynth.pipelines.qwen_image import QwenImagePipeline, ModelConfig

qwen = os.environ['QWEN_DIR']
te_shards = [f"{qwen}/text_encoder/model-{i:05d}-of-00004.safetensors" for i in range(1, 5)]

pipe = QwenImagePipeline.from_pretrained(
    torch_dtype=torch.bfloat16, device='cpu',
    model_configs=[ModelConfig(path=te_shards)],
    tokenizer_config=ModelConfig(f"{qwen}/tokenizer"),
)
unit = pipe.units[-1]  # QwenImageUnit_PromptEmbedder
mode = os.environ.get('ODTSR_ENCODE_DEVICE', 'cpu')

if mode == 'gpu_stream':
    # Full-precision bf16, but only ONE transformer layer lives on the GPU at a time.
    # CPU copies stay as the loader produced them (mmap-friendly), so RAM stays low
    # and VRAM peak is ~0.5-1 GB - fits on a T4 even with the DiT already loaded.
    te = pipe.text_encoder
    inner = getattr(te, 'model', te)
    # Qwen2.5-VL bundles TWO transformer stacks: a vision tower (`.visual.blocks`,
    # 32 small ViT blocks, hidden=1280) and the language decoder (`.language_model.layers`,
    # 28 LARGE layers, hidden=3584, intermediate=18944). Picking "the ModuleList with the
    # most elements" grabs the vision stack (32 > 28) by mistake: THAT gets streamed,
    # while the much heavier language layers fall into "everything else" and get
    # permanently parked on the GPU -> OOM (~15 GB doesn't fit on a T4). Pick by total
    # parameter count instead, which finds the language decoder regardless of naming.
    # The vision tower is also unused for text-only prompt encoding (no pixel_values
    # are ever passed in this pipeline), so it's simplest and safest to leave it on
    # the CPU untouched rather than stream or park it.
    visual_module = getattr(inner, 'visual', None)
    layers, layers_param_count = None, -1
    for _n, _m in te.named_modules():
        if isinstance(_m, torch.nn.ModuleList):
            pc = sum(p.numel() for p in _m.parameters())
            if pc > layers_param_count:
                layers, layers_param_count = _m, pc
    assert layers is not None, 'transformer layer list not found'
    print(f'text encoder: streaming target = {len(layers)} layers, '
          f'{layers_param_count/1e9:.2f}B params '
          f'(vision tower left on CPU, unused for text-only encoding)', flush=True)

    def _attach_streaming(layer):
        tensors = [p for p in layer.parameters(recurse=True)]
        tensors += [b for _, b in layer.named_buffers(recurse=True) if b is not None]
        cpu_copies = [t.data for t in tensors]
        def pre(module, args, kwargs):
            for t in tensors:
                t.data = t.data.to('cuda:0', non_blocking=True)
        def post(module, args, output):
            for t, cpu in zip(tensors, cpu_copies):
                t.data = cpu
            return output
        layer.register_forward_pre_hook(lambda m, a, k: pre(m, a, k), with_kwargs=True)
        layer.register_forward_hook(post)

    layer_ids = {id(l) for l in layers}
    for l in layers:
        _attach_streaming(l)
    # everything not in the streamed stack AND not part of the (unused) vision tower
    # is small (embeddings, norms, rotary cache, lm head) -> park it on the GPU permanently
    skip_ids = {id(m) for m in visual_module.modules()} if visual_module is not None else set()
    for m in te.modules():
        if id(m) in layer_ids or id(m) in skip_ids:
            continue
        for p in list(m.parameters(recurse=False)):
            if p.device.type == 'cpu':
                p.data = p.data.to('cuda:0')
        for bn, b in list(m.named_buffers(recurse=False)):
            if b is not None and b.device.type == 'cpu':
                setattr(m, bn, b.to('cuda:0'))
    # inputs/hidden states must land on cuda; layer params stream in on demand
    def _mv(obj, device):
        if torch.is_tensor(obj): return obj.to(device)
        if isinstance(obj, tuple): return tuple(_mv(o, device) for o in obj)
        if isinstance(obj, list): return [_mv(o, device) for o in obj]
        if isinstance(obj, dict): return {k: _mv(v, device) for k, v in obj.items()}
        return obj
    te.register_forward_pre_hook(
        lambda m, a, k: (_mv(a, 'cuda:0'), _mv(k, 'cuda:0')), with_kwargs=True)
    pipe.device = 'cuda:0'
    print(f'text encoder: bf16 STREAMING on GPU, {len(layers)} layers, '
          f'{torch.cuda.memory_allocated(0)/1e9:.1f} GB resident VRAM', flush=True)

elif mode == 'gpu4bit':
    import bitsandbytes as bnb
    te = pipe.text_encoder
    def _quantize_te(root):
        n_q = 0
        for name, child in list(root.named_children()):
            if type(child) is torch.nn.Linear and child.weight.numel() >= 1_000_000:
                w = child.weight.data.to(torch.bfloat16)
                new = bnb.nn.Linear4bit(child.in_features, child.out_features,
                                        bias=child.bias is not None,
                                        compute_dtype=torch.bfloat16, quant_type='nf4')
                new.weight = bnb.nn.Params4bit(data=w.contiguous(), requires_grad=False,
                                               quant_type='nf4')
                if child.bias is not None:
                    new.bias = torch.nn.Parameter(child.bias.data.to(torch.bfloat16),
                                                  requires_grad=False)
                setattr(root, name, new.to('cuda:0'))
                n_q += 1
            else:
                n_q += _quantize_te(child)
        return n_q
    n_q = _quantize_te(te)
    te.to('cuda:0')
    gc.collect(); torch.cuda.empty_cache()
    pipe.device = 'cuda:0'
    print(f'text encoder: {n_q} Linear -> NF4 on GPU (LOSSY), '
          f'{torch.cuda.memory_allocated(0)/1e9:.1f} GB VRAM', flush=True)

cache_path = os.environ['PROMPT_CACHE_PATH']
cache = torch.load(cache_path, map_location='cpu') if os.path.exists(cache_path) else {}
for p in json.loads(os.environ['PROMPTS_JSON']):
    print('encode:', repr(p), flush=True)
    with torch.no_grad():
        out = unit.process(pipe, prompt=p)
    cache[p] = {k: (v.detach().to('cpu') if torch.is_tensor(v) else v) for k, v in out.items()}
torch.save(cache, cache_path)
print('saved:', list(cache.keys()), flush=True)
'''
    script_path = '/content/encode_prompts.py'
    open(script_path, 'w').write(encoder_script)
    env = dict(os.environ,
               ODTSR_DIR=ODTSR_DIR, QWEN_DIR=QWEN_DIR,
               PROMPT_CACHE_PATH=PROMPT_CACHE_PATH,
               PROMPTS_JSON=json.dumps(missing),
               ODTSR_ENCODE_DEVICE=enc_mode,
               CUDA_VISIBLE_DEVICES=('' if enc_mode == 'cpu' else '0'))
    r = subprocess.run(['python', script_path], env=env, capture_output=True, text=True)
    if show_log:
        print(r.stdout[-1500:])
    if r.returncode != 0:
        print(r.stdout[-1500:])  # always show on failure, even with show_log off
        print(r.stderr[-4000:])
        raise RuntimeError('Prompt encoding failed - see the log above.')

PROMPT_CACHE = torch.load(PROMPT_CACHE_PATH, map_location='cpu')

# ============================================================
# 2) Load model onto the GPU — only once per session
# ============================================================
# A model already sitting in GPU memory does NOT get re-quantized just because you
# changed `quantization`/`SENSITIVE_INT8` above — the load below is skipped entirely
# unless we detect one of them changed. Without this check, changing settings and
# re-running this cell would silently keep the OLD (differently-quantized) DiT on the
# GPU: the printed Config summary would say one thing, the live model another.
_prev_load_key = globals().get('_LOADED_QUANT_MODE')
_cur_load_key = (QUANT_MODE, SENSITIVE_INT8 if QUANT_MODE == 'int4' else None)
if 'model' in globals() and _prev_load_key != _cur_load_key:
    print(f'Quantization settings changed ({_prev_load_key!r} -> {_cur_load_key!r}) - '
          'forcing a full reload (old GPU weights are being freed)...')
    for _nm in ('model', 'dit', 'blocks', 'prompt_unit'):
        if _nm in globals():
            del globals()[_nm]
    gc.collect(); torch.cuda.empty_cache()

if 'model' not in globals():
    # ODTSR_FP8='0' -> NO dtype conversion on CPU. The safetensors shards are bf16 on
    # disk, safe_open().get_tensor() memory-maps them (zero-copy), .to(bf16) is a no-op
    # and load_state_dict(assign=True) keeps the mmap. The 40 GB DiT therefore uses
    # ~0 RSS (only reclaimable page cache) instead of the ~20 GB fp8 copy that used to
    # crash free Colab (12.7 GB RAM). Quantization below then streams block-by-block
    # to the GPU, touching one layer at a time.
    os.environ['ODTSR_FP8'] = '0'

    os.chdir(f'{ODTSR_DIR}/examples/qwen_image')
    from generator import Generator

    transformer_shards = [f'{QWEN_DIR}/transformer/diffusion_pytorch_model-{i:05d}-of-00009.safetensors' for i in range(1, 10)]
    weights_json = json.dumps([transformer_shards, f'{QWEN_DIR}/vae/diffusion_pytorch_model.safetensors'])

    print('Loading DiT + VAE on CPU via zero-copy mmap (bf16, low RAM)... '
          '(~5-15 min, RAM should stay under ~5 GB. Only happens once per session.)')
    # Generator's __init__ prints one line per Linear layer it casts/wraps (hundreds
    # of "[cast] .../[lora] ..." lines) - genuinely useful for debugging, but noise
    # otherwise. Captured and only shown if show_log, or always on failure.
    _load_buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(_load_buf if not show_log else sys.stdout):
            model = Generator(
                torch_dtype=torch.bfloat16,
                pretrained_weights=weights_json,
                tokenizer_path=f'{QWEN_DIR}/tokenizer',
                learning_rate=0,
                use_gradient_checkpointing=False,
                pretrained_ckpt_path_gen=None,
            )
    except Exception:
        print(_load_buf.getvalue())
        raise
    model.pipe.requires_grad_(False)
    model.eval()
    gc.collect()

    from model_training.train import LinearFP8Wrapper, DualLoRALinear
    # Defensive no-op with ODTSR_FP8='0' (nothing is fp8 anymore); kept in case FP8 is re-enabled.
    # fp8 -> bf16 for everything that is NOT a Linear weight (norms, embeddings, ...)
    n_fixed = 0
    for m in model.pipe.dit.modules():
        if isinstance(m, (torch.nn.Linear, LinearFP8Wrapper)):
            continue
        for p in m.parameters(recurse=False):
            if p.dtype == torch.float8_e4m3fn:
                p.data = p.data.to(torch.bfloat16); n_fixed += 1
        for bname, buf in list(m.named_buffers(recurse=False)):
            if buf is not None and buf.dtype == torch.float8_e4m3fn:
                setattr(m, bname, buf.to(torch.bfloat16)); n_fixed += 1
    print(f'fp8 -> bf16 for non-Linear weights: {n_fixed} tensors')

    # --- ODTSR checkpoint MUST be loaded BEFORE quantization (keys match fp8/bf16 layout) ---
    print('Loading the ODTSR checkpoint...')
    assert 'net_dis' not in os.path.basename(CKPT_PATH), (
        f'CKPT_PATH points to the discriminator: {CKPT_PATH}. '
        'Re-run the Install cell — it must download weight.pth.'
    )
    try:  # mmap=True keeps the 2.5 GB ckpt on disk instead of RAM
        state_dict = torch.load(CKPT_PATH, map_location='cpu', mmap=True)
    except Exception:
        state_dict = torch.load(CKPT_PATH, map_location='cpu')
    if isinstance(state_dict, dict) and 'state_dict' in state_dict and not torch.is_tensor(state_dict['state_dict']):
        state_dict = state_dict['state_dict']

    # Apply by ASSIGNMENT, not load_state_dict(): the base DiT params are read-only
    # mmap tensors now, so the in-place copy_() inside load_state_dict would fail —
    # and assignment also avoids materializing anything extra in RAM.
    params = dict(model.named_parameters())
    bufs = dict(model.named_buffers())
    unexpected_k, n_applied = [], 0
    for k, v in state_dict.items():
        if k in params:
            p = params[k]
            p.data = v if v.dtype == p.dtype else v.to(p.dtype)
            n_applied += 1
        elif k in bufs:
            b = bufs[k]
            b.data = v if v.dtype == b.dtype else v.to(b.dtype)
            n_applied += 1
        else:
            unexpected_k.append(k)
    missing_k = [k for k in params if k not in state_dict]
    print(f'checkpoint apply: applied={n_applied}, missing={len(missing_k)}, unexpected={len(unexpected_k)}')
    assert len(unexpected_k) == 0, (
        f'{len(unexpected_k)} checkpoint keys did not match the model, examples: {unexpected_k[:5]}'
    )
    lora_missing = [k for k in missing_k if 'lora' in k.lower()]
    assert len(lora_missing) == 0, (
        f'LoRA weights were NOT loaded ({len(lora_missing)} keys), examples: {lora_missing[:5]}'
    )
    n_lora_loaded = sum(1 for k in state_dict if 'lora' in k.lower())
    print(f'OK: generator checkpoint applied, LoRA tensors loaded: {n_lora_loaded}')
    del state_dict, params, bufs
    gc.collect(); torch.cuda.empty_cache()

    dit = model.pipe.dit
    blocks = dit.transformer_blocks
    n = len(blocks)
    print(f'DiT: {n} blocks, mode: {QUANT_MODE}')

    # ------------------------------------------------------------------
    # Placement / quantization  (v2: mixed INT8/NF4)
    # ------------------------------------------------------------------
    if QUANT_MODE in ('int4', 'int8'):
        import bitsandbytes as bnb

        MIN_QUANT_NUMEL = 1_000_000
        _SENS = [] if SENSITIVE_INT8 == 'none' else SENSITIVE_INT8.split('+')
        _stats = {'int8': 0, 'nf4': 0, 'bf16': 0}

        class _CastLinear8bit(torch.nn.Module):
            """Linear8bitLt computes in fp16 - cast bf16 activations in and out."""
            def __init__(self, inner):
                super().__init__()
                self.inner = inner
            def forward(self, x):
                return self.inner(x.to(torch.float16)).to(x.dtype)

        def _bf16_linear(w, b):
            out_f, in_f = w.shape
            lin = torch.nn.Linear(in_f, out_f, bias=b is not None, dtype=torch.bfloat16)
            lin.weight = torch.nn.Parameter(w, requires_grad=False)
            if b is not None:
                lin.bias = torch.nn.Parameter(b, requires_grad=False)
            _stats['bf16'] += 1
            return lin.to('cuda:0')

        def _int8_linear(w, b):
            out_f, in_f = w.shape
            new = bnb.nn.Linear8bitLt(in_f, out_f, bias=b is not None,
                                      has_fp16_weights=False, threshold=6.0)
            new.weight = bnb.nn.Int8Params(data=w.to(torch.float16).contiguous(),
                                           requires_grad=False, has_fp16_weights=False)
            if b is not None:
                new.bias = torch.nn.Parameter(b.to(torch.float16), requires_grad=False)
            _stats['int8'] += 1
            return _CastLinear8bit(new.to('cuda:0'))

        def _nf4_linear(w, b):
            out_f, in_f = w.shape
            new = bnb.nn.Linear4bit(in_f, out_f, bias=b is not None,
                                    compute_dtype=torch.bfloat16, quant_type='nf4')
            # compress_statistics=True (double quantization): quantizes the NF4 scale
            # factors themselves, saving ~0.4 bit/param across the whole DiT (~1 GB on
            # this model). Slightly less accurate than False, but on a 14.5-15.6 GB T4
            # this GB is the difference between SENSITIVE_INT8='img_mod' fitting at all
            # and OOMing at block ~51/60 - measured on this exact GPU/model.
            new.weight = bnb.nn.Params4bit(data=w.contiguous(), requires_grad=False,
                                           quant_type='nf4', compress_statistics=True)
            if b is not None:
                new.bias = torch.nn.Parameter(b, requires_grad=False)
            _stats['nf4'] += 1
            return new.to('cuda:0')

        def _make_quant(weight_any, bias_any, layer_path=''):
            w = weight_any.data.to(torch.bfloat16)
            b = bias_any.data.to(torch.bfloat16) if bias_any is not None else None
            if w.numel() < MIN_QUANT_NUMEL:
                return _bf16_linear(w, b)
            sensitive = any(t in layer_path for t in _SENS)
            if QUANT_MODE == 'int8' or sensitive:
                return _int8_linear(w, b)
            return _nf4_linear(w, b)

        def _base_linear_quant(self, x):
            return self.linear(x)

        def _quantize_tree(root, path=''):
            n_q = 0
            for name, child in list(root.named_children()):
                p = f'{path}.{name}' if path else name
                if isinstance(child, DualLoRALinear):
                    child.linear = _make_quant(child.linear.weight, child.linear.bias, p)
                    child._base_linear = types.MethodType(_base_linear_quant, child)
                    for l in (child.lora_A1, child.lora_B1, child.lora_A2, child.lora_B2):
                        l.to(device='cuda:0', dtype=torch.bfloat16)
                    n_q += 1
                elif isinstance(child, LinearFP8Wrapper) or type(child) is torch.nn.Linear:
                    setattr(root, name, _make_quant(child.weight, child.bias, p))
                    n_q += 1
                else:
                    n_q += _quantize_tree(child, p)
            return n_q

        def _unwrap_leftover_fp8(root, path=''):
            # txt_in / norm_out.linear / proj_out передавались в _quantize_tree КАК root,
            # а он проверяет только детей -> обёртки оставались. Разворачиваем в bf16
            # (они маленькие, bf16 = полное качество).
            n = 0
            for name, child in list(root.named_children()):
                p = f'{path}.{name}' if path else name
                if isinstance(child, LinearFP8Wrapper):
                    w = child.weight.data.to(torch.bfloat16)
                    b = child.bias.data.to(torch.bfloat16) if child.bias is not None else None
                    setattr(root, name, _bf16_linear(w, b))
                    n += 1
                elif not isinstance(child, (DualLoRALinear, bnb.nn.Linear4bit,
                                            bnb.nn.Linear8bitLt, _CastLinear8bit)):
                    n += _unwrap_leftover_fp8(child, p)
            return n

        # Reserve real VRAM for inference (VAE tiles, DiT activations, attention buffers) -
        # quantizing block-by-block with NO safety margin either fits with ~0 GB left for
        # Run (immediate OOM there) or OOMs mid-load. Now: quantize + keep on GPU as many
        # blocks as fit while leaving INFERENCE_HEADROOM_GB free: once that budget would be
        # violated, remaining blocks are left in BF16 and CPU-offloaded (same mechanism as
        # the pure-BF16 branch below) instead of quantized - slower for just those blocks,
        # but correct and OOM-safe, and still better quality than forcing everything NF4.
        # 3.5 GB matches the reserve the pure-BF16-offload branch below already uses
        # successfully (vram_free - 3.5).
        INFERENCE_HEADROOM_GB = 3.5

        def _move(obj, device):
            if torch.is_tensor(obj):
                return obj.to(device)
            if isinstance(obj, tuple):
                return tuple(_move(o, device) for o in obj)
            if isinstance(obj, list):
                return [_move(o, device) for o in obj]
            if isinstance(obj, dict):
                return {k: _move(v, device) for k, v in obj.items()}
            return obj

        def _make_pre_hook(device):
            def hook(module, args, kwargs):
                return _move(args, device), _move(kwargs, device)
            return hook

        def _attach_device_hook(module):
            """Move this module's inputs to wherever ITS OWN parameters live, right
            before its forward call. Required on EVERY block/tail-module here - GPU
            AND CPU alike - because temb/hidden-states are shared globally across all
            60 blocks. If even one upstream module (e.g. time_text_embed) ends up
            CPU-offloaded, downstream GPU-resident blocks would otherwise receive a
            CPU tensor and crash with a device-mismatch RuntimeError. Mirrors the
            pure-BF16-offload branch below, which hooks every block for the same
            reason regardless of where each one lives."""
            try:
                dev = next(module.parameters()).device
            except StopIteration:
                return
            module.register_forward_pre_hook(_make_pre_hook(dev), with_kwargs=True)

        def _attach_cpu_offload(module):
            """Leave `module` in BF16 on CPU; stream its inputs to wherever it lives on
            each forward call (mirrors the pure-BF16-offload branch below)."""
            module.to('cpu')
            _attach_device_hook(module)

        print(f'Quantizing DiT: mod-layers -> INT8 ({SENSITIVE_INT8}), rest -> NF4, '
              f'block by block (~5-10 min), reserving {INFERENCE_HEADROOM_GB:.1f} GB '
              'headroom for inference...')
        total_q = 0
        n_offloaded = 0
        _budget_hit = False
        for i, b in enumerate(blocks):
            _free_gb = torch.cuda.mem_get_info()[0] / 1e9
            if _budget_hit or _free_gb < INFERENCE_HEADROOM_GB:
                if not _budget_hit:
                    print(f'  headroom budget reached at block {i}/{n} ({_free_gb:.2f} GB free) - '
                          f'blocks {i}..{n-1} will be BF16 CPU-offloaded instead of quantized.')
                    _budget_hit = True
                _attach_cpu_offload(b)
                n_offloaded += 1
                continue
            total_q += _quantize_tree(b)
            b.to('cuda:0')
            _attach_device_hook(b)
            if i % 10 == 9:
                gc.collect(); torch.cuda.empty_cache()
                print(f'  blocks {i+1}/{n}, VRAM {torch.cuda.memory_allocated(0)/1e9:.1f} GB, '
                      f'{torch.cuda.mem_get_info()[0]/1e9:.2f} GB free')

        def _safe_to_cuda(module):
            # Same headroom check as the block loop - the tail modules (time_text_embed,
            # img_in, txt_in, norm_out, proj_out) are individually small, but "small"
            # still isn't free once only ~100 MB is left: route every remaining GPU
            # placement through here instead of an unconditional .to('cuda:0').
            _free_gb = torch.cuda.mem_get_info()[0] / 1e9
            if _free_gb < INFERENCE_HEADROOM_GB:
                _attach_cpu_offload(module)
            else:
                module.to('cuda:0')
                _attach_device_hook(module)

        if _budget_hit:
            # Headroom was already spent inside the block loop - don't even attempt to
            # quantize the tail modules (that itself needs a transient GPU buffer), just
            # leave them BF16 CPU-offloaded like the tail DiT blocks.
            print('  headroom already spent on the block loop - leaving time_text_embed/'
                  'img_in/txt_in/norm_out/proj_out as BF16 CPU-offload too.')
            for name in ('time_text_embed', 'img_in', 'txt_in', 'norm_out', 'proj_out'):
                _attach_cpu_offload(getattr(dit, name))
            n_unwrapped = 0
        else:
            total_q += _quantize_tree(dit.time_text_embed, 'time_text_embed')
            if isinstance(dit.img_in, DualLoRALinear):
                dit.img_in.linear = _make_quant(dit.img_in.linear.weight,
                                                dit.img_in.linear.bias, 'img_in')
                dit.img_in._base_linear = types.MethodType(_base_linear_quant, dit.img_in)
            n_unwrapped = _unwrap_leftover_fp8(dit)
            for name in ('time_text_embed', 'img_in', 'txt_in', 'norm_out', 'proj_out'):
                _safe_to_cuda(getattr(dit, name))

        for name in ('pos_embed', 'txt_norm'):
            _safe_to_cuda(getattr(dit, name))
        # NOTE: no blanket dit.to('cuda:0') here anymore - that would drag the
        # CPU-offloaded tail blocks back onto the GPU and defeat the whole point.
        dit.proj_out.register_forward_hook(lambda mod, args, out: _move(out, 'cuda:0'))
        print(f'Quantized {total_q} linear layers '
              f'(int8={_stats["int8"]}, nf4={_stats["nf4"]}, bf16={_stats["bf16"]}, '
              f'fp8-wrappers unwrapped={n_unwrapped}); {n_offloaded}/{n} blocks '
              f'BF16 CPU-offloaded for lack of headroom.')

    else:  # bf16 + CPU offload — weights are bf16 now (mmap), NOT fp8
        vram_free = torch.cuda.get_device_properties(0).total_memory / 1e9
        per_block_gb = 0.7  # bf16 storage (weights stay bf16 — no fp8 conversion on free Colab)
        on_gpu = max(4, min(n, int((vram_free - 3.5) / per_block_gb)))
        block_devices = ['cuda:0' if i < on_gpu else 'cpu' for i in range(n)]
        print(f'BF16 offload: {on_gpu}/{n} blocks on GPU, {n - on_gpu} on CPU '
              '(CPU blocks stay mmap-backed = low RAM, but inference is slow; '
              'prefer INT4 on a T4).')

        for name in ('pos_embed', 'time_text_embed', 'txt_norm', 'img_in', 'txt_in'):
            getattr(dit, name).to('cuda:0')
        for i, b in enumerate(blocks):
            b.to(block_devices[i])
            if i % 10 == 9:
                gc.collect(); torch.cuda.empty_cache()
        dit.norm_out.to('cuda:0')
        dit.proj_out.to('cuda:0')

        def _move(obj, device):
            if torch.is_tensor(obj):
                return obj.to(device)
            if isinstance(obj, tuple):
                return tuple(_move(o, device) for o in obj)
            if isinstance(obj, list):
                return [_move(o, device) for o in obj]
            if isinstance(obj, dict):
                return {k: _move(v, device) for k, v in obj.items()}
            return obj

        def _make_pre_hook(device):
            def hook(module, args, kwargs):
                return _move(args, device), _move(kwargs, device)
            return hook

        hooked = list(blocks) + [dit.norm_out, dit.proj_out, dit.img_in, dit.txt_in,
                                 dit.txt_norm, dit.time_text_embed]
        for m in hooked:
            try:
                dev = next(m.parameters()).device
            except StopIteration:
                continue
            m.register_forward_pre_hook(_make_pre_hook(dev), with_kwargs=True)
        dit.proj_out.register_forward_hook(lambda mod, args, out: _move(out, 'cuda:0'))

    model.pipe.vae.to('cuda:0')
    model.pipe.new_vae.to('cuda:0')
    model.device = torch.device('cuda:0')
    model.pipe.device = 'cuda:0'
    gc.collect(); torch.cuda.empty_cache()

    _free_after, _total_after = torch.cuda.mem_get_info()
    print(f'GPU headroom after load: {_free_after/1e9:.2f} GB free / {_total_after/1e9:.2f} GB total')
    if _free_after < 1.5e9:
        print(f'WARN: only {_free_after/1e9:.2f} GB free after loading - Run will likely OOM on '
              'the first VAE/DiT tile regardless of tile size. Try a lighter setting '
              '(SENSITIVE_INT8=\'none\', or QUANT_MODE=INT4) and reload.')
    print(f'GPU 0: {torch.cuda.memory_allocated(0)/1e9:.1f} GB used')
    import psutil
    print(f'RAM: {psutil.virtual_memory().percent}% used')
    print('Model loaded.\n')
    _LOADED_QUANT_MODE = _cur_load_key
else:
    print(f'Model already loaded (quantization={_LOADED_QUANT_MODE}) — skipping reload.\n')

# prompt embedder always re-bound to the (possibly refreshed) cache
prompt_unit = model.pipe.units[-1]
def cached_process(self, pipe, prompt=None, **kwargs):
    key = (prompt or '').strip()
    if key not in PROMPT_CACHE:
        raise RuntimeError(f'Prompt {key!r} is not cached — re-run this cell.')
    return {k: (v.to(pipe.device) if torch.is_tensor(v) else v) for k, v in PROMPT_CACHE[key].items()}
prompt_unit.process = types.MethodType(cached_process, prompt_unit)

# ============================================================
# 3) Tiling helpers: adaptive padding + VAE tile-size override
# ============================================================
def adaptive_pad(img, tilesize, stride):
    w, h = img.size
    pad_h = (tilesize - h) if h <= tilesize else ((h - tilesize + stride - 1) // stride) * stride + tilesize - h
    pad_w = (tilesize - w) if w <= tilesize else ((w - tilesize + stride - 1) // stride) * stride + tilesize - w
    new_img = Image.new(img.mode, (w + pad_w, h + pad_h), color=0)
    new_img.paste(img, (0, 0))
    return new_img

def _patch_vae_tiles(vae):
    # Whenever the pipeline calls vae.encode/decode with tiled=True, silently swap in
    # the tile sizes from current_tiles() (Config dropdowns or the Auto-tune result).
    if vae is None or getattr(vae, '_tile_patched', False):
        return
    _orig_encode = vae.encode
    def _encode(x, **kw):
        if kw.get('tiled'):
            _t = current_tiles()
            kw['tile_size'] = _t['vae_enc']  # pixels
            kw['tile_stride'] = max(_t['vae_enc'] - _t['overlap'] * 8, _t['vae_enc'] // 2)
        return _orig_encode(x, **kw)
    vae.encode = _encode
    if hasattr(vae, 'decode'):
        _orig_decode = vae.decode
        def _decode(x, **kw):
            if kw.get('tiled'):
                _t = current_tiles()
                kw['tile_size'] = _t['vae_dec']  # latent units
                kw['tile_stride'] = max(_t['vae_dec'] - _t['overlap'], _t['vae_dec'] // 2)
            return _orig_decode(x, **kw)
        vae.decode = _decode
    vae._tile_patched = True

_patch_vae_tiles(model.pipe.vae)
_patch_vae_tiles(getattr(model.pipe, 'new_vae', None))
print('VAE tiling override active:', current_tiles())


In [ ]:
#@title ##**Run** { display-mode: "form" }
# Upscales every queued image using the model from the Load Model cell and the tile
# sizes from Auto-tune tiles (if it ran) or Config. No before/after preview here —
# see the Visualization cell.
import os, gc, glob, time, torch
from PIL import Image

assert 'model' in globals(), 'Run the Load Model cell first.'

# Defensive VRAM cleanup: a previous OOM/exception in this session can leave GPU
# tensors alive via Jupyter's exception traceback (IPython caches sys.last_traceback,
# which pins every local variable of every stack frame - including any tensor that was
# mid-construction when the error happened). Plain gc.collect()/empty_cache() often
# can't reclaim that memory until the traceback itself is dropped, so clear it first.
import sys
sys.last_traceback = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    _free, _total = torch.cuda.mem_get_info()
    print(f'GPU memory: {_free/1e9:.2f} GB free / {_total/1e9:.2f} GB total')

TILES = current_tiles()
LAT_TILE = TILES['latent']
LAT_STRIDE = LAT_TILE - TILES['overlap']
print(f"tiles ({'auto-tuned' if AUTO_TILES else 'from Config'}): "
      f"vae_enc={TILES['vae_enc']}px, vae_dec={TILES['vae_dec']}, "
      f"latent={LAT_TILE}, overlap={TILES['overlap']} (stride {LAT_STRIDE})")

run_scale = float(scale)
run_prompt = prompt.strip()
run_negative = negative_prompt.strip()
for _p in {run_prompt, run_negative, ''}:
    assert _p in PROMPT_CACHE, f'Prompt {_p!r} is not encoded — re-run the Load Model cell.'
run_fidelity = float(fidelity)
run_cfg = float(cfg)
torch.manual_seed(int(seed))
os.environ['ODTSR_SEED'] = str(int(seed))

exts = ('*.png', '*.jpg', '*.jpeg', '*.webp')
extra_folder = globals().get('extra_folder', '') or ''
paths = []
for e in exts:
    paths += glob.glob(os.path.join(INPUT_DIR, e))
    if extra_folder:
        paths += glob.glob(os.path.join(extra_folder, '**', e), recursive=True)
paths = sorted(set(paths))
assert paths, 'No images found! Upload some in Upload Images.'
print(f'Images: {len(paths)}, scale x{run_scale}, fidelity={run_fidelity}, cfg={run_cfg}, '
      f'prompt: {run_prompt!r}, negative: {run_negative!r}\n')

from wavelet_color_fix import adain_color_fix, wavelet_color_fix

RESULTS = []  # (input_path, output_path) pairs for this run — used by Visualization

for idx, img_path in enumerate(paths):
    name = os.path.splitext(os.path.basename(img_path))[0]
    res_path = os.path.join(OUTPUT_DIR, f'{name}_x{run_scale:g}.png')
    print(f'[{idx+1}/{len(paths)}] {os.path.basename(img_path)}', end=' ')
    t0 = time.perf_counter()

    img = Image.open(img_path).convert('RGB')
    w, h = img.size
    w_dst, h_dst = round(w * run_scale), round(h * run_scale)
    upsampled = img.resize((w_dst, h_dst), Image.BICUBIC)
    padded = adaptive_pad(upsampled, tilesize=LAT_TILE * 8, stride=LAT_STRIDE * 8)

    with torch.no_grad():
        res = model.infer(
            prompt=run_prompt,
            negative_prompt=run_negative,
            condition_image=padded,
            cfg_scale=run_cfg,
            fidelity=run_fidelity,
            tiled=True,
            tile_size=LAT_TILE,
            tile_stride=LAT_STRIDE,
        )

    res = res.crop((0, 0, w_dst, h_dst))
    if color_fix == 'wavelet':
        res = wavelet_color_fix(res, upsampled)
    elif color_fix == 'adain':
        res = adain_color_fix(res, upsampled)
    res.save(res_path)
    torch.cuda.empty_cache()

    RESULTS.append((img_path, res_path))
    print(f'-> {res_path} ({time.perf_counter()-t0:.1f} s)')

print(f'\nDone. {len(RESULTS)} image(s) saved to {OUTPUT_DIR}. '
      'Open the Visualization cell to compare before/after.')

In [ ]:
#@title ##**Visualization** { display-mode: "form" }
import os, base64
from io import BytesIO
import PIL.Image
from IPython.display import HTML, display

if 'RESULTS' not in globals() or not RESULTS:
    raise RuntimeError('No results found — run the Run cell first.')

# Extensions the browser's canvas.toDataURL can encode directly.
# Anything else (bmp, tiff, ...) is saved as PNG on download.
CANVAS_MIME = {
    '.png': 'image/png',
    '.jpg': 'image/jpeg',
    '.jpeg': 'image/jpeg',
    '.webp': 'image/webp',
}

def image_to_base64(image):
    buffered = BytesIO()
    image.save(buffered, format='PNG')
    return base64.b64encode(buffered.getvalue()).decode('utf-8')

def fit_box(width, height, max_width, target_height=None):
    # Only used to size the on-screen comparison box; the embedded image data
    # itself stays full resolution so the download matches the real upscale.
    if width > max_width:
        height = int(height * max_width / width)
        width = max_width
    if target_height is not None and height != target_height:
        width = int(width * target_height / height)
        height = target_height
    return width, height

for idx, (img_path, res_path) in enumerate(RESULTS):
    try:
        filename = os.path.basename(img_path)
        out_filename = os.path.basename(res_path)

        image_before = PIL.Image.open(img_path)
        image_after = PIL.Image.open(res_path)

        if image_before.mode != 'RGB':
            image_before = image_before.convert('RGB')
        if image_after.mode != 'RGB':
            image_after = image_after.convert('RGB')

        # Display-only box size (full-res pixel data is embedded either way)
        max_width = min(image_after.size[0], 700)
        display_w, display_h = fit_box(*image_after.size, max_width)

        before_base64 = image_to_base64(image_before)
        after_base64 = image_to_base64(image_after)

        # Use the result's original format for the download if the browser can encode it via
        # canvas, otherwise fall back to PNG.
        out_ext = os.path.splitext(res_path)[1].lower()
        download_mime = CANVAS_MIME.get(out_ext, 'image/png')
        download_ext = out_ext if out_ext in CANVAS_MIME else '.png'

        uid = f"cmp{idx}"
        base_name = os.path.splitext(filename)[0]

        html_code = f"""
        <style>
            .download-btn-{uid} {{
                background-color: #28a745;
                color: white;
                border: none;
                border-radius: 4px;
                padding: 5px 10px;
                cursor: pointer;
                display: flex;
                align-items: center;
                gap: 6px;
                font-family: sans-serif;
                font-size: 12px;
                font-weight: bold;
                flex-shrink: 0;
                transition: background-color 0.15s ease-in-out;
            }}
            .download-btn-{uid}:hover {{
                background-color: #218838;
            }}
            .download-btn-{uid}:active {{
                background-color: #1e7e34;
            }}
        </style>
        <div style="width: {display_w}px; margin-bottom: 30px;">
            <div style="margin-bottom:6px; font-family: sans-serif; font-size: 13px; font-weight: bold;">
                {filename} &rarr; {out_filename}
            </div>
            <div id="{uid}" style="position: relative; width: 100%; height: {display_h}px;">
                <div style="position: relative; width: 100%; height: 100%; overflow: hidden;">
                    <img class="img-before" src="data:image/png;base64,{before_base64}" style="position: absolute; width: 100%; height: 100%;">
                    <div class="clip-wrap" style="position: absolute; width: 100%; height: 100%; overflow: hidden; clip-path: inset(0 0 0 50%);">
                        <canvas class="canvas-after" style="position: absolute; width: 100%; height: 100%; opacity: 1;"></canvas>
                    </div>
                </div>
                <div class="slider" style="position: absolute; top: 0; bottom: 0; width: 4px; background: white; cursor: ew-resize; left: 50%; box-shadow: 0 0 5px rgba(0,0,0,0.5);">
                    <div style="position: absolute; top: 50%; transform: translateY(-50%); width: 20px; height: 20px; background: white; border-radius: 50%; left: -8px;"></div>
                </div>
                <div style="position: absolute; top: 8px; left: 8px; color: white; font-family: sans-serif; font-size: 14px; text-shadow: 0 1px 3px rgba(0,0,0,0.8);">Before</div>
                <div style="position: absolute; top: 8px; right: 8px; color: white; font-family: sans-serif; font-size: 14px; text-shadow: 0 1px 3px rgba(0,0,0,0.8);">After</div>
            </div>

            <div style="display: flex; flex-direction: column; gap: 8px; margin-top: 8px; font-family: sans-serif; font-size: 13px; color: inherit; min-width: 360px;">
                <div style="display: flex; align-items: center; gap: 10px;">
                    <span style="white-space: nowrap; flex-shrink: 0; min-width: 90px;">Blend original:</span>
                    <input type="range" class="blend-slider" min="0" max="100" value="100" step="1" style="flex: 1;">
                    <span class="blend-value" style="min-width: 45px; text-align: right; flex-shrink: 0;">100%</span>
                    <div style="width: 100px; flex-shrink: 0;"></div>
                </div>

                <div style="display: flex; align-items: center; gap: 10px;">
                    <span style="white-space: nowrap; flex-shrink: 0; min-width: 90px;">Sharpness:</span>
                    <input type="range" class="sharpness-slider" min="0.0" max="7.0" value="0.0" step="0.1" style="flex: 1;">
                    <span class="sharpness-value" style="min-width: 45px; text-align: right; flex-shrink: 0;">0.0</span>
                    <button class="download-btn-{uid}" title="Download blended and sharpened image ({download_ext}, full resolution)" style="width: 100px;">
                        <svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" style="display: inline-block;">
                            <path d="M21 15v4a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-4"></path>
                            <polyline points="7 10 12 15 17 10"></polyline>
                            <line x1="12" y1="15" x2="12" y2="3"></line>
                        </svg>
                        Download
                    </button>
                </div>
            </div>
        </div>
        <script>
            (function() {{
                const root = document.getElementById('{uid}');
                const slider = root.querySelector('.slider');
                const container = root.querySelector('div:nth-child(2)');
                const clipDiv = root.querySelector('.clip-wrap');
                let isDragging = false;

                slider.addEventListener('mousedown', (e) => {{ isDragging = true; e.preventDefault(); }});
                document.addEventListener('mouseup', () => {{ isDragging = false; }});
                document.addEventListener('mousemove', (e) => {{
                    if (!isDragging) return;
                    const rect = container.getBoundingClientRect();
                    let x = e.clientX - rect.left;
                    if (x < 0) x = 0;
                    if (x > rect.width) x = rect.width;
                    const percentage = (x / rect.width) * 100;
                    slider.style.left = percentage + '%';
                    clipDiv.style.clipPath = `inset(0 0 0 ${{percentage}}%)`;
                }});

                slider.addEventListener('touchstart', (e) => {{ isDragging = true; e.preventDefault(); }});
                document.addEventListener('touchend', () => {{ isDragging = false; }});
                document.addEventListener('touchmove', (e) => {{
                    if (!isDragging) return;
                    const rect = container.getBoundingClientRect();
                    let x = e.touches[0].clientX - rect.left;
                    if (x < 0) x = 0;
                    if (x > rect.width) x = rect.width;
                    const percentage = (x / rect.width) * 100;
                    slider.style.left = percentage + '%';
                    clipDiv.style.clipPath = `inset(0 0 0 ${{percentage}}%)`;
                }});
            }})();

            (function() {{
                const root = document.getElementById('{uid}');
                const wrapper = root.parentElement;
                const blendSlider = wrapper.querySelector('.blend-slider');
                const blendValue = wrapper.querySelector('.blend-value');
                const sharpnessSlider = wrapper.querySelector('.sharpness-slider');
                const sharpnessValue = wrapper.querySelector('.sharpness-value');

                const canvasAfter = root.querySelector('.canvas-after');
                const downloadBtn = wrapper.querySelector('.download-btn-{uid}');

                const downloadMime = "{download_mime}";
                const downloadExt = "{download_ext}";

                const imgBeforeSource = new Image();
                imgBeforeSource.src = "data:image/png;base64,{before_base64}";

                const imgAfterSource = new Image();
                imgAfterSource.src = "data:image/png;base64,{after_base64}";

                let originalImgData = null;
                let smoothedPixels = null;
                let ctx = null;

                function getSmoothedPixels(imgData) {{
                    const w = imgData.width;
                    const h = imgData.height;
                    const src = imgData.data;
                    const dst = new Uint8ClampedArray(src.length);
                    dst.set(src);

                    const kernel = [
                        1, 1, 1,
                        1, 5, 1,
                        1, 1, 1
                    ];
                    const divisor = 13;

                    for (let y = 1; y < h - 1; y++) {{
                        for (let x = 1; x < w - 1; x++) {{
                            let r = 0, g = 0, b = 0;
                            for (let ky = -1; ky <= 1; ky++) {{
                                const rowOffset = (y + ky) * w * 4;
                                for (let kx = -1; kx <= 1; kx++) {{
                                    const idx = rowOffset + (x + kx) * 4;
                                    const weight = kernel[(ky + 1) * 3 + (kx + 1)];
                                    r += src[idx] * weight;
                                    g += src[idx + 1] * weight;
                                    b += src[idx + 2] * weight;
                                }}
                            }}
                            const dstIdx = (y * w + x) * 4;
                            dst[dstIdx]     = Math.min(255, Math.max(0, r / divisor));
                            dst[dstIdx + 1] = Math.min(255, Math.max(0, g / divisor));
                            dst[dstIdx + 2] = Math.min(255, Math.max(0, b / divisor));
                        }}
                    }}
                    return dst;
                }}

                function applySharpness(factor) {{
                    if (!originalImgData || !smoothedPixels) return;

                    const w = canvasAfter.width;
                    const h = canvasAfter.height;

                    if (factor === 1.0) {{
                        ctx.putImageData(originalImgData, 0, 0);
                        return;
                    }}

                    const src = originalImgData.data;
                    const smoothed = smoothedPixels;

                    const outputData = ctx.createImageData(w, h);
                    const dst = outputData.data;

                    for (let i = 0; i < src.length; i += 4) {{
                        dst[i]     = Math.min(255, Math.max(0, smoothed[i] * (1 - factor) + src[i] * factor));
                        dst[i + 1] = Math.min(255, Math.max(0, smoothed[i + 1] * (1 - factor) + src[i + 1] * factor));
                        dst[i + 2] = Math.min(255, Math.max(0, smoothed[i + 2] * (1 - factor) + src[i + 2] * factor));
                        dst[i + 3] = src[i + 3];
                    }}
                    ctx.putImageData(outputData, 0, 0);
                }}

                imgAfterSource.onload = () => {{
                    // Backing-store resolution = the real upscaled image (imgAfterSource is
                    // full-res); CSS width/height:100% on the canvas scale it down for the
                    // compact on-screen preview only. This is what makes the download full-res.
                    canvasAfter.width = imgAfterSource.naturalWidth;
                    canvasAfter.height = imgAfterSource.naturalHeight;
                    ctx = canvasAfter.getContext('2d');

                    ctx.drawImage(imgAfterSource, 0, 0);
                    originalImgData = ctx.getImageData(0, 0, canvasAfter.width, canvasAfter.height);

                    smoothedPixels = getSmoothedPixels(originalImgData);

                    applySharpness(0.0);
                }};

                sharpnessSlider.addEventListener('input', (e) => {{
                    const val = parseFloat(e.target.value);
                    sharpnessValue.textContent = val.toFixed(1);
                    applySharpness(val);
                }});

                blendSlider.addEventListener('input', (e) => {{
                    const val = e.target.value;
                    blendValue.textContent = val + '%';
                    canvasAfter.style.opacity = val / 100;
                }});

                downloadBtn.addEventListener('click', () => {{
                    // w/h come from canvasAfter's backing store, i.e. the full upscaled resolution
                    const canvas = document.createElement('canvas');
                    const w = canvasAfter.width;
                    const h = canvasAfter.height;
                    canvas.width = w;
                    canvas.height = h;

                    const downloadCtx = canvas.getContext('2d');

                    downloadCtx.drawImage(imgBeforeSource, 0, 0, w, h);

                    const opacity = parseFloat(blendSlider.value) / 100;
                    downloadCtx.globalAlpha = opacity;
                    downloadCtx.drawImage(canvasAfter, 0, 0, w, h);

                    const link = document.createElement('a');
                    link.download = 'blended_{base_name}' + downloadExt;
                    link.href = canvas.toDataURL(downloadMime, 0.95);
                    link.click();
                }});
            }})();
        </script>
        """

        display(HTML(html_code))

    except Exception as e:
        print(f'Error processing {img_path} and {res_path}: {e}')

In [ ]:
#@title ##**Download Results** { display-mode: "form" }
# One click: a single image downloads as-is; several images are zipped first.
# google.colab.files.download streams straight from disk — no base64 tricks needed.
import os, zipfile
from google.colab import files

results = sorted(f for f in os.listdir(OUTPUT_DIR) if f.lower().endswith('.png'))
assert results, 'No results — run the Run cell first.'

total_mb = sum(os.path.getsize(os.path.join(OUTPUT_DIR, f)) for f in results) / 1e6
print(f'{len(results)} image(s), ~{total_mb:.1f} MB total.')
print('(Also always available in the Files panel on the left -> /content/outputs.)')

if len(results) == 1:
    files.download(os.path.join(OUTPUT_DIR, results[0]))
else:
    zpath = '/content/odtsr_results.zip'
    if os.path.exists(zpath):
        os.remove(zpath)
    # ZIP_STORED: PNGs are already compressed, DEFLATE would just burn CPU
    with zipfile.ZipFile(zpath, 'w', compression=zipfile.ZIP_STORED) as zf:
        for f in results:
            zf.write(os.path.join(OUTPUT_DIR, f), arcname=f)
    print(f'Packed {zpath} ({os.path.getsize(zpath)/1e6:.1f} MB) — starting download...')
    files.download(zpath)